In [ ]:
import os, subprocess, sys, json, time
os.environ["PYTHONUTF8"]="1"
print(subprocess.run([sys.executable,"-m","pip","install","-q","flopscope>=0.12.1,<0.13","whestbench>=0.16.1,<0.17"],capture_output=True,text=True).stdout[-500:])
import flopscope; from importlib.metadata import version; print("flopscope", version("flopscope"), "whestbench", version("whestbench"))
import os; print("cpus", os.cpu_count()); print(subprocess.run(["free","-g"],capture_output=True,text=True).stdout)

In [ ]:
import glob
print(subprocess.run(["find","/kaggle/input","-maxdepth","3"],capture_output=True,text=True).stdout[:2000])
hits = glob.glob("/kaggle/input/**/variants.json", recursive=True); print("variants.json at", hits)
INPUT = os.path.dirname(hits[0])
VARIANTS = json.load(open(f"{INPUT}/variants.json"))   # {"tag": {"file": "...py", "env": {...}}}
N_MLPS = int(os.environ.get("N_MLPS", "10"))
print(VARIANTS, N_MLPS)

In [ ]:
# Download only the mini split (prepared Arrow fast path) into the HF cache
os.environ["HF_XET_HIGH_PERFORMANCE"]="1"
t0=time.time()
r=subprocess.run(["whest","dataset","download","aicrowd/arc-whestbench-public-2026","--revision","v2-phase2","--split","mini"],capture_output=True,text=True)
print(r.stdout[-800:], r.stderr[-800:]); print("download s", time.time()-t0)

In [ ]:
results = {}
for tag, spec in VARIANTS.items():
    env = dict(os.environ); env.update({k: str(v) for k, v in spec.get("env", {}).items()}); env["PYTHONWARNINGS"]="ignore"
    out = f"/kaggle/working/run_{tag}.json"
    cmd = ["whest","run","--estimator",f"{INPUT}/{spec['file']}","--dataset","hf://aicrowd/arc-whestbench-public-2026@v2-phase2","--split","mini",
           "--n-mlps",str(N_MLPS),"--runner","local","--wall-time-limit","900","--no-residual-wall-time-limit","--format","json","--profile"]
    t0=time.time(); r=subprocess.run(cmd, env=env, capture_output=True, text=True)
    open(out,"w").write(r.stdout)
    try:
        res=json.loads(r.stdout)["results"]
        per=[(m["mlp_name"], m["flops_used"]/2**41, m["wall_time_s"], m["residual_wall_time_s"], m["final_layer_mse"], m["error_code"] if "error_code" in m else None) for m in res["per_mlp"]]
        results[tag]={k:res[k] for k in ("adjusted_final_layer_score","final_layer_mse","all_layers_mse","mean_compute_utilization","n_failed_mlps","failure_breakdown")}
        results[tag]["per_mlp"]=per; results[tag]["elapsed_s"]=time.time()-t0
        print(tag, json.dumps(results[tag], indent=None)[:1500])
    except Exception as e:
        print(tag, "FAILED to parse:", e, r.stdout[-1500:], r.stderr[-3000:])
json.dump(results, open("/kaggle/working/summary.json","w"), indent=1)
base = next((k for k in results if k.startswith("base")), None)
if base:
    import math
    b = {r[0]: r[4] for r in results[base]["per_mlp"]}
    for tag, r in results.items():
        pairs = [(row[4], b[row[0]]) for row in r["per_mlp"] if row[0] in b]
        ratios = [x / y for x, y in pairs]
        gm = math.exp(sum(map(math.log, ratios)) / len(ratios))
        wins = sum(1 for x, y in pairs if x < y)
        print(f"{tag:12s} raw {r['final_layer_mse']:.4e} util {r['mean_compute_utilization']:.4f} failed {r['n_failed_mlps']} | vs {base}: geo-mean ratio {gm:.4f}, better on {wins}/{len(pairs)}")